# This notebook contains heavy imports and calculations, please do not run on a regular CPU

(but for example on Colab)

<br>

# Training and Fine-Tuning BERT for Classification
## Classifying tweets as offensive or not

This notebook demonstrates how to train and fine-tune a BERT-style model for classification with the HuggingFace `transformers` library.

We will fine-tune [`jhu-clsp/mmBERT-small`](https://huggingface.co/jhu-clsp/mmBERT-small) on the `tweets_dataset` (the `cardiffnlp/tweet_eval` "offensive" subset) that you already downloaded and saved in `first_full_pipeline.ipynb`, with the goal of predicting whether a tweet is:

-   `non-offensive`
-   `offensive`

If you haven't run `first_full_pipeline.ipynb` yet, do that first so `data/tweets_dataset` exists on disk.

**Basic steps involved in using BERT and HuggingFace:**
- Load the dataset's train/validation/test subsets.
- Convert your data into a format that BERT can process.
- Create dataset objects by joining your data and labels.
- Load the pre-trained BERT model.
- Refine the model by training it on your training data.
- Use the model to make predictions and assess its performance on your test data.


<br><br>

## **Import necessary Python libraries and modules**

Next, we will import necessary Python libraries and modules.

In [ ]:
import os
import random
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils import compute_sample_weight
import torch
from datasets import load_dataset, load_from_disk
from transformers import Trainer, TrainingArguments



from collections import defaultdict

sns.set_theme(style='ticks', font_scale=1.2)

%matplotlib inline
import matplotlib.pyplot as plt

os.environ["WANDB_DISABLED"] = "true"

<br>

## **Read in data**

This will read in the `tweets_dataset` you already downloaded and saved with `first_full_pipeline.ipynb`. That dataset already comes split into `train`, `validation`, and `test` subsets, so we don't need to split it ourselves.


In [ ]:
#For Colab, run this cell to download the dataset (if you haven't already done so) and save it to your Google Drive. You will be prompted to authenticate your Google account.:
#Skip this step if you have already downloaded the dataset and saved it to your Google Drive. 
from google.colab import drive

## Mount Google Drive
drive.mount('/content/drive')
os.chdir('/content/drive/') #os.chdir() — changes the working directory so relative paths work correctly
#make sure to change the path to where you saved the tweets_dataset on your Google Drive
#set datadir to your drive-location where you saved the dataset (only if you didn't save it in '/content/drive/MyDrive/'):
#for example:
datadir = "MyDrive/"
#
## Load dataset
tweets = load_dataset("cardiffnlp/tweet_eval", "offensive")
#
## Save to Google Drive
tweets.save_to_disk("/content/drive/My Drive/tweets_dataset")

Note. In Google Colab: If you had already downloaded the data and saved it to your drive, you don't need to download again, just import and mount your drive and load the dataset from your Google Drive


In [ ]:
#Load the dataset from Google Drive, or adjust the location to where you saved the dataset on your local machine

tweets = load_from_disk(os.path.join(datadir, "tweets_dataset"))

train_tweets = tweets["train"].to_pandas()
val_tweets = tweets["validation"].to_pandas()
test_tweets = tweets["test"].to_pandas()

X_train, y_train = train_tweets['text'].to_list(), train_tweets['label'].to_list()
X_val, y_val = val_tweets['text'].to_list(), val_tweets['label'].to_list()
X_test, y_test = test_tweets['text'].to_list(), test_tweets['label'].to_list()

In [ ]:
print(f"We have {len(X_train)} train examples, {len(X_val)} validation examples, and {len(X_test)} test examples.")

Here's an example of a training text and training label:

In [ ]:
X_train[0], y_train[0]

<br><br>

## **Implementing a Baseline Model using Logistic Regression**

In this step, we train and evaluate a basic TF-IDF baseline model with logistic regression. We observe a performance that is clearly better than random guessing. We will now check if BERT can outperform this strong baseline!

In [ ]:
vectorizer = TfidfVectorizer()
Xtrain = vectorizer.fit_transform(X_train)
Xtest = vectorizer.transform(X_test)

We train a logistic regression model from scikit-learn on the tweet training data, and then we use the trained model to make predictions on our test set.

In [ ]:
model = LogisticRegression(max_iter=1000).fit(Xtrain, y_train)
predictions = model.predict(Xtest)

We can leverage the `classification_report` function provided by scikit-learn to assess the performance of the logistic regression model in terms of its ability to predict tweet offensiveness that match the actual labels.

In [ ]:
print(classification_report(y_test, predictions, target_names=tweets["test"].features["label"].names))

What do you think of this model? Not too bad for a baseline model, right? Lets see whether we can improve this using BERT.

(But note that below we will give our BERT much less data to train on, which is in favor of the baseline. At the same time, we could further optimize the baseline with pre-processing etc as you saw this morning.)

## Encode data for BERT

To prepare our data for use with BERT, we need to encode the texts and labels in a way that the model can understand. Here are the steps we'll follow:

1. Convert the labels from strings to integers if needed.

2. Tokenize the texts, which involves breaking them up into individual words, and then convert the words into "word pieces" that can be matched with their corresponding embedding vectors.

3. Make sure all texts are of equal lenght: Truncate texts that are longer than 200 tokens, or pad texts that are shorter than 200 tokens with a special padding token.

4. Add special tokens to the beginning and end of each document, including a start token, a separator between sentences, and a padding token as necessary.


We will be using the `AutoTokenizer.from_pretrained()` module from HuggingFace library to encode our texts. This module will handle all the encoding for us, including breaking word tokens into word pieces, truncating to 200 tokens, and adding padding and special BERT tokens.

In [ ]:
# Ideally, we want to run our code on CUDA (NVIDIA GPUs using the program management system) or MPS (Apple Silicon GPUs).

# Check if there is a GPU available...
if torch.cuda.is_available():
    # Tell PyTorch to use the CUDA GPU.
    device_name = torch.device("cuda")
    print('There are %d CUDA GPU(s) available.' % torch.cuda.device_count())
    print('We will use the CUDA GPU:', torch.cuda.get_device_name(0))

# Check if MPS is available...
elif torch.backends.mps.is_available():
    # Tell PyTorch to use the MPS GPU.
    device_name = torch.device("mps")
    print('MPS is available.')
    print('We will use the MPS GPU.')

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device_name = torch.device("cpu")

In [ ]:
#We will be using mmBERT-small, a small multilingual BERT-style (ModernBERT) model -- specifically the "jhu-clsp/mmBERT-small" model. Check out Hugging Face's documentation for more information on the different BERT models.
model_name = 'jhu-clsp/mmBERT-small'

# We set the maximum number of tokens in each document to be 200. Tweets are short (typically ~20–40 tokens), so this comfortably covers them (mmBERT itself supports much longer sequences).
max_length = 200

# We define the directory where we'll save our trained model. You can choose any name for the directory.
save_directory = '/tweets_dataset/fine_tuned_model'

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

In this section, we will generate a mapping of our tweet labels to integer keys. We begin by extracting the unique labels from our dataset and create a dictionary that associates each label with an integer.

In [ ]:
label_names = tweets["test"].features["label"].names
label2id = {label: idx for idx, label in enumerate(label_names)} #Note the curly brackets '{}', this is a dictionary rather than a list
id2label = {idx: label for idx, label in enumerate(label_names)}

In [ ]:
print(label_names)
print(label2id)
print(id2label)

In [ ]:
label2id.keys()

In [ ]:
id2label.keys()

Since fine-tuning on the full training set takes about an hour on Colab, we will take a sample here for illustration.

Note that this will affect performance...

In [ ]:
from sklearn.model_selection import train_test_split

# --- Stratified subsample of the training set for faster fine-tuning ---
# Preserves the offensive / non-offensive ratio of the full training set.
# Only the TRAIN set is shrunk -- val and test stay full so metrics remain comparable.
SUBSAMPLE_TRAIN = True  #you can change this to False if you want to finetune on the full sample instead
train_sample_size = 3000    # absolute number of training examples to keep
# train_sample_size = 0.2   # ...or a fraction, e.g. 0.2 for 20%
seed = 42

if SUBSAMPLE_TRAIN:
    X_train, _, y_train, _ = train_test_split(    #we use the train_test_split function here, this is a good one to remember, since not all datasets come with a predefined split
        X_train, y_train,
        train_size=train_sample_size,
        stratify=y_train,                         #we want to be careful about the distribution of our classes, here we state we want to preserve the distribution of the full training sample
        random_state=seed,
    )
    print(f"Subsampled training set to {len(X_train)} examples")
    print("Class balance:",
          pd.Series(y_train).value_counts(normalize=True).round(3).to_dict())

Now let's encode our texts and labels!

In [ ]:
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=max_length)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=max_length)
test_encodings  = tokenizer(X_test, truncation=True, padding=True, max_length=max_length)

# The dataset's labels are already integer-encoded (0/1), matching id2label/label2id above, so no lookup is needed here.
train_labels_encoded = y_train
val_labels_encoded = y_val
test_labels_encoded  = y_test

**Examine a tweet in the training set after encoding**

In [ ]:
' '.join(train_encodings[0].tokens[0:100])

**Examine a tweet in test set after encoding**

In [ ]:
' '.join(test_encodings[0].tokens[0:100])

**Examine the training labels after encoding**

In [ ]:
set(train_labels_encoded)

**Examine the test labels after encoding**

In [ ]:
set(test_labels_encoded)

<br><br>

## **Create a custom Torch dataset by following these steps:**

Here we combine the encoded labels and texts into dataset objects. We use the custom Torch `MyDataSet` class to make a `train_dataset` object from  the `train_encodings` and `train_labels_encoded`. We also make a `val_dataset`, `test_dataset` object from `test_encodings` and `val_encodings`, and `val_labels_encoded` and `test_labels_encoded`.


In [ ]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = MyDataset(train_encodings, train_labels_encoded)
val_dataset = MyDataset(val_encodings, val_labels_encoded)
test_dataset = MyDataset(test_encodings, test_labels_encoded)

**Examine a tweet in the Torch `training_dataset` after encoding**

In [ ]:
' '.join(train_dataset.encodings[0].tokens[0:100])

**Examine a tweet in the Torch `test_dataset` after encoding**

In [ ]:
' '.join(test_dataset.encodings[1].tokens[0:100])

In [ ]:
len(id2label)

<br><br>

## **Initialize the pre-trained BERT model**

We load a pre-trained mmBERT model and transfer it to CUDA (or MPS/CPU) for efficient computation.

**Note**: If you intend to repeat the fine-tuning process after previously executing the subsequent cells, ensure that you re-run this cell to reload the original pre-trained model before commencing the fine-tuning again.

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(id2label)).to(device_name)

<br>
<br>

## **Configure the parameters required for fine-tuning BERT**

Fine-tuning is adding a small classifier on top of the pre-trained model and nudging the whole network's weights toward your task.

The following parameters are crucial for fine-tuning BERT and will be specified in the HuggingFace TrainingArguments objects that we will subsequently pass to the HuggingFace Trainer object. While there are numerous other arguments, we'll focus on the fundamental ones and some common pitfalls.

When fine-tuning your own model, it's critical to experiment with these parameters to identify the optimal configuration for your specific dataset.

| Parameter                     | Explanation                                                                                                                          |
|-------------------------------|--------------------------------------------------------------------------------------------------------------------------------------|
| `num_train_epochs`            | The total number of training epochs. This refers to how many times the entire dataset will be processed. Too many epochs can lead to overfitting.|
| `per_device_train_batch_size` | The batch size per device during training.                                                                                           |
| `per_device_eval_batch_size`  | The batch size for evaluation.                                                                                                      |
| `warmup_steps`                | The number of warmup steps for the learning rate scheduler. A smaller value is recommended for small datasets.                         |
| `weight_decay`                | The strength of weight decay, which reduces the size of weights, similar to regularization.                                          |
| `output_dir`                  | The directory where the fine-tuned model, checkpoints, and logs will be saved.                                                     |
| `logging_steps`               | How often to print logging output. This enables us to terminate training early if the loss is not decreasing.                        |
| `eval_strategy`                | Evaluates while training so that we can monitor accuracy improvements.                                                              |


<br><br>

## **Fine-tune the BERT model**

Initially, we define a custom evaluation function that returns the accuracy and F1 of the model. However, this function can be modified to return other metrics such as precision, recall, or any other desired evaluation metric.

In [ ]:
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1) #classify the document with the class with the highest predicted value. Note: here we only have two labels, but this procedure is suited for multiclass classificaton tasks as well
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'macro_f1': macro_f1}

Then we create a HuggingFace `Trainer` object using the `TrainingArguments` object that we created above. We also send our `compute_metrics` function to the `Trainer` object, along with our test and train datasets.


## **optimize your model based on a metric you select**
Note: You can also use GridSearch to identify the optimal configuration. However, be aware that finetuning multiple times with different parameter combinations can be extremely resource-intensive.

In [ ]:
metric_name = 'macro_f1' # you can chance this for `accuracy` etc, according to the function `compute_metrics`

In [ ]:
# Instantiate an object of the TrainingArguments class with the following parameters:
training_args = TrainingArguments(

    # Number of training epochs
    num_train_epochs=2, #note default is 3, but I noted overfitting in while testing, so I reduced it to 2. Note, we could also train for 5 epochs to see if we can improve performance (but note overfitting -> we will discuss this tomorrow)

    # Batch size for training
    per_device_train_batch_size=8,

    # Batch size for evaluation
    per_device_eval_batch_size=8,

    # Learning rate for optimization -> how much to change the weights of the model during training (the smaller the learning rate, the smaller the changes to the weights)
    learning_rate=5e-5, #Decrease (1e-5 to 2e-5) if performance is unstable; increase slightly if learning is too slow

    # Load the best model at the end of training
    load_best_model_at_end=True, #Note this is not necessarily the model from the last epoch, but the best model according to the metric_for_best_model below, helpful if the model is overfitting and performance on the validation set is decreasing

    # Metric used for selecting the best model
    metric_for_best_model=metric_name,

    # Number of warmup steps for the optimizer
    warmup_steps=0,

    # L2 regularization weight decay # How much to penalize large weights
    weight_decay=0.01,

    # Directory to save the fine-tuned model and configuration files
    output_dir='/tweets_dataset/results', #Note, you might have to create this folder first at the correct location, or change the path here to where you want the results to be stored

    # Log results every n steps
    logging_steps=20,

    # Strategy for evaluating the model during training
    eval_strategy='steps',

    # Set a seed for reproducibility, but note that this does not guarantee full reproducibility, since some operations are non-deterministic on GPUs (e.g., cuDNN)
    seed=42
)

In [ ]:
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=val_dataset,            # validation dataset, used to monitor training and pick the best checkpoint -- the test set stays held out for final evaluation
    compute_metrics=compute_metrics      # our custom evaluation function
)

Time to finally fine-tune!

Be patient; if you've set everything in Colab to use GPUs, then it should only take about 8 minutes to run, but if you're running on CPU, it can take hours.

After every 20 steps (as we specified in the TrainingArguments object), the trainer will output the current state of the model, including the training loss, validation loss, and accuracy (from our `compute_metrics` function).

You should see the loss going down and the accuracy going up. If instead they are staying the same or oscillating, you probably need to change the fine-tuning parameters.

In [ ]:
trainer.train()

<br><br>

## **Save fine-tuned model**

The following cell will save the model and its configuration files to a directory in Colab. To preserve this model for future use, you should download the model to your computer.

In [ ]:
trainer.save_model(save_directory)  

(Optional) If you've already fine-tuned and saved the model, you can reload it using the following line. You don't have to run fine-tuning every time you want to evaluate.

In [ ]:
# model = AutoModelForSequenceClassification.from_pretrained(save_directory)

<br><br>

## **Evaluate fine-tuned model on the validation set**

The following function of the `Trainer` object will run the built-in evaluation, including our `compute_metrics` function.

In [ ]:
trainer.evaluate()

<br><br>

## **Evaluate fine-tuned model on the test set**

We may desire a more detailed evaluation of the model, hence we extract the predicted labels.

In [ ]:
predicted_results = trainer.predict(test_dataset)

In [ ]:
predicted_results.predictions.shape

In [ ]:
predicted_label_ids = predicted_results.predictions.argmax(-1).flatten().tolist()  # Get the highest probability prediction, flattened into a 1D list
predicted_labels = [id2label[l] for l in predicted_label_ids]  # Convert from integers back to strings for readability
true_labels = [id2label[l] for l in y_test]  # Convert the true integer labels the same way, so both sides are comparable

In [ ]:
len(predicted_labels)

In [ ]:
print(classification_report(true_labels, predicted_labels))

<br><br>

## **Extracting Correct and Incorrect Classifications for Analysis**

Now that we have obtained the predicted labels, let's perform some analysis.

The fine-tuning and extraction of predicted labels using BERT is now complete. You can use the predicted labels just like you would with any other classification model. Here are some examples.

To start, let's print out some example predictions that were correct.

In [ ]:
for _true_label, _predicted_label, _text in random.sample(list(zip(true_labels, predicted_labels, X_test)), 20):
  if _true_label == _predicted_label:
    print('LABEL:', _true_label)
    print('TWEET TEXT:', _text)
    print()

Now let's print out some misclassifications.

In [ ]:
for _true_label, _predicted_label, _text in random.sample(list(zip(true_labels, predicted_labels, X_test)), 80):
  if _true_label != _predicted_label:
    print('TRUE LABEL:', _true_label)
    print('PREDICTED LABEL:', _predicted_label)
    print('TWEET TEXT:', _text)
    print()

Finally, let's create some heatmaps to examine misclassification patterns. We can use these patterns to see whether the model confuses offensive and non-offensive tweets in a particular direction.

In [ ]:
from collections import Counter

# Count the number of classifications for each label pair
label_classifications = Counter(zip(true_labels, predicted_labels))

# Convert the counts to a DataFrame and pivot to wide format
df_wide = pd.DataFrame(label_classifications, index=['Number of Classifications']).T.reset_index()
df_wide.columns = ['True Label', 'Predicted Label', 'Number of Classifications']
df_wide = df_wide.pivot_table(index='True Label', columns='Predicted Label', values='Number of Classifications', fill_value=0)

# Plot the results
plt.figure(figsize=(9,7))
sns.heatmap(df_wide, linewidths=1, cmap='viridis', annot=True, fmt='.0f', cbar=False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Looks good! We can see that overall, our model is assigning the correct labels for most tweets.

Now, let's remove the diagonal from the plot to highlight the misclassifications.

In [ ]:
label_classifications_dict = defaultdict(int)
for _true_label, _predicted_label in zip(true_labels, predicted_labels):
  if _true_label != _predicted_label: # Remove the diagonal to highlight misclassifications
    label_classifications_dict[(_true_label, _predicted_label)] += 1

dicts_to_plot = []
for (_true_label, _predicted_label), _count in label_classifications_dict.items():
  dicts_to_plot.append({'True Label': _true_label,
                        'Predicted Label': _predicted_label,
                        'Number of Classifications': _count})

df_to_plot = pd.DataFrame(dicts_to_plot)
df_wide = df_to_plot.pivot_table(index='True Label',
                                 columns='Predicted Label',
                                 values='Number of Classifications')

plt.figure(figsize=(9,7))
sns.heatmap(df_wide, linewidths=1, cmap='viridis')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()